# SHAP-Analyse – autosklearn

docker cp ROOTFOLDER\persistent_predict_server.py DOCKERID:/workspace/persistent_predict_server.py


In [ ]:
import os
import json
import subprocess
import uuid
import numpy as np
import matplotlib.pyplot as plt
import shap

print("shap version: ", shap.__version__)

# ======================================================================
# CONFIG - HIER ANPASSEN
# ======================================================================
FRAMEWORK = 'auto-sklearn'          # 'auto-sklearn' | 'auto-pytorch' | 'autogluon'

# Container-ID des laufenden Original-Containers (docker ps -> CONTAINER ID)
# Pro Framework ggf. unterschiedlich!
CONTAINER_ID = '36b2e233bcf1'       # <-- HIER DEINE ECHTE ID EINTRAGEN

# Kernel läuft IM shap-Container. Der sieht das Projekt unter /workspace.
HOST_BASE = r"E:\StudiumMasterarbeit"
CONT_BASE = "/data"

# Persistenter Server IM Container (absoluter Pfad)
PERSISTENT_SERVER_IN_CONTAINER = {
    'auto-sklearn': '/data/persistent_predict_server.py',
    'auto-pytorch': '/workspace/persistent_predict_server.py',
    'autogluon':    '/workspace/persistent_predict_server_autogluon.py',
}[FRAMEWORK]

# Verzeichnisse (Host-seitig, wie der Kernel sie sieht)
SAVED_MODELS_DIRS = {
    'auto-sklearn': 'saved_models/auto-sklearn',
    'auto-pytorch': 'saved_models/auto-pytorch',
    'autogluon':    'saved_models/autogluon',
}
SAVED_MODELS_DIR = SAVED_MODELS_DIRS[FRAMEWORK]
OUTPUT_DIR = 'shap_results'

# Temp-Ordner fuer den Austausch Host<->Container. Muss unter HOST_BASE liegen,
# damit der Container ihn auch sieht (ueber den Volume-Mount).
TMP_DIR = os.path.join(HOST_BASE, 'tmp_shap_bridge')
os.makedirs(TMP_DIR, exist_ok=True)

N_BACKGROUND = 50
N_EXPLAIN    = 30
SKIP_IF_RESULTS_EXIST = True
# ======================================================================


def container_path(host_path):
    """Mappt einen Host-Pfad (E:\\StudiumMasterarbeit\\...) auf den
    entsprechenden Container-Pfad (/workspace\\...)."""
    host_path = os.path.abspath(host_path)
    base = os.path.abspath(HOST_BASE)
    if not os.path.normcase(host_path).startswith(os.path.normcase(base)): # needed for lowercase-uppercase differences on Windows
        raise ValueError(f"Pfad liegt nicht unter {base}: {host_path}")
    rel = os.path.relpath(host_path, base)
    rel = rel.replace('\\', '/')
    return f"{CONT_BASE}/{rel}"


def list_available_tasks(framework):
    base_dir = SAVED_MODELS_DIRS[framework]
    if not os.path.isdir(base_dir):
        print(f"Kein Verzeichnis gefunden: {base_dir}")
        return []
    return sorted(
        d for d in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, d))
    )


def run_shap_for_task(framework, task_name,
                      n_background=N_BACKGROUND, n_explain=N_EXPLAIN):
    print(f"\n{'='*60}")
    print(f"SHAP-Analyse: [{framework}] {task_name}")
    print(f"{'='*60}")

    task_dir = os.path.join(SAVED_MODELS_DIRS[framework], task_name)
    out_dir  = os.path.join(OUTPUT_DIR, framework, task_name)
    os.makedirs(out_dir, exist_ok=True)

    with open(os.path.join(task_dir, "feature_names.json")) as f:
        feature_names = json.load(f)

    X_train = np.load(os.path.join(task_dir, "X_train.npy"))
    X_test  = np.load(os.path.join(task_dir, "X_test.npy"))

    background = shap.sample(X_train,
                             min(n_background, len(X_train)),
                             random_state=42)
    X_explain = X_test[:min(n_explain, len(X_test))]
    max_evals = max(500, 2 * len(feature_names) + 1)

    print(f"  Hintergrund: {len(background)} | Erklären: {len(X_explain)} "
          f"| Merkmale: {len(feature_names)} | max_evals: {max_evals}")

    model_host_path      = os.path.join(task_dir, "model.joblib")
    model_container_path = container_path(model_host_path)

    proc = subprocess.Popen(
        [
            "docker", "exec", "-i", CONTAINER_ID,
            "python3", "-u",
            PERSISTENT_SERVER_IN_CONTAINER,
            model_container_path,
        ],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
    )

    try:
        first = proc.stdout.readline().strip()
        if first != "READY":
            err = proc.stderr.read() if proc.stderr else ""
            raise RuntimeError(
                f"Handshake fehlgeschlagen: {first!r}\nstderr: {err[:2000]}"
            )
        print("  Predict-Server bereit.")

        def predict(X):
            uid = uuid.uuid4().hex
            host_in  = os.path.join(TMP_DIR, f"in_{uid}.npy")
            host_out = os.path.join(TMP_DIR, f"out_{uid}.npy")
            try:
                np.save(host_in, np.asarray(X))
                proc.stdin.write(
                    f"{container_path(host_in)}\t{container_path(host_out)}\n"
                )
                proc.stdin.flush()
                reply = proc.stdout.readline().strip()
                if reply != "OK":
                    err = proc.stderr.read() if proc.stderr else ""
                    raise RuntimeError(
                        f"Predict-Server: {reply!r}\nstderr: {err[:2000]}"
                    )
                return np.load(host_out)
            finally:
                for p in (host_in, host_out):
                    if os.path.exists(p):
                        try:
                            os.remove(p)
                        except OSError:
                            pass

        explainer = shap.Explainer(
            predict, background, feature_names=feature_names, seed=42
        )
        shap_values = explainer(X_explain, max_evals=max_evals)

        y_pred_explained = predict(X_explain)
        base_values = np.asarray(shap_values.base_values)
        efficiency_gap = y_pred_explained - (
            shap_values.values.sum(axis=1) + base_values
        )
        print(f"  Efficiency-Check: mean|gap| = "
              f"{np.mean(np.abs(efficiency_gap)):.6f} "
              f"(mean|y| = {np.mean(np.abs(y_pred_explained)):.6f})")

    finally:
        try:
            proc.stdin.write("EXIT\n")
            proc.stdin.flush()
        except Exception:
            pass
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.terminate()
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                proc.kill()

    # Plots
    plt.figure()
    shap.summary_plot(shap_values, X_explain,
                      feature_names=feature_names,
                      plot_type='bar', show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'shap_summary_bar.png'), dpi=150)
    plt.close()

    plt.figure()
    shap.summary_plot(shap_values, X_explain,
                      feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'shap_summary_beeswarm.png'), dpi=150)
    plt.close()

    # Rohdaten
    np.save(os.path.join(out_dir, "shap_values.npy"), shap_values.values)
    np.save(os.path.join(out_dir, "X_explained.npy"), X_explain)
    np.save(os.path.join(out_dir, "base_values.npy"), base_values)
    np.save(os.path.join(out_dir, "y_pred_explained.npy"), y_pred_explained)
    with open(os.path.join(out_dir, "feature_names.json"), "w") as f:
        json.dump(list(feature_names), f)

    print(f"  Gespeichert: {out_dir}")
    return shap_values


def task_already_done(framework, task_name):
    return os.path.exists(
        os.path.join(OUTPUT_DIR, framework, task_name, "shap_values.npy")
    )


# ======================================================================
# AUSFUEHRUNG
# ======================================================================
tasks = list_available_tasks(FRAMEWORK)
print(f"\n{'#'*70}")
print(f"# SHAP FUER ALLE AUFGABEN: [{FRAMEWORK}] - {len(tasks)} Aufgaben")
print(f"{'#'*70}")
for t in tasks:
    print(f"  - {t}")

status = {}
for i, task_name in enumerate(tasks):
    print(f"\n[{i+1}/{len(tasks)}] {FRAMEWORK} / {task_name}")
    if SKIP_IF_RESULTS_EXIST and task_already_done(FRAMEWORK, task_name):
        print(f"  -> UEBERSPRUNGEN (Ergebnis existiert bereits)")
        status[task_name] = 'skipped'
        continue
    try:
        run_shap_for_task(FRAMEWORK, task_name)
        status[task_name] = 'ok'
    except Exception as e:
        print(f"  FEHLER: {e}")
        status[task_name] = f'failed: {e}'

status_path = os.path.join(OUTPUT_DIR, FRAMEWORK, '_run_status.json')
os.makedirs(os.path.dirname(status_path), exist_ok=True)
with open(status_path, 'w') as f:
    json.dump(status, f, indent=2)

n_ok      = sum(1 for v in status.values() if v == 'ok')
n_skipped = sum(1 for v in status.values() if v == 'skipped')
n_failed  = sum(1 for v in status.values() if str(v).startswith('failed'))
print(f"\n{'='*60}")
print(f"ZUSAMMENFASSUNG [{FRAMEWORK}]: {n_ok} neu, {n_skipped} uebersprungen, "
      f"{n_failed} fehlgeschlagen")
print(f"Status: {status_path}")